In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
project_path = os.path.join(os.getcwd(),'..','..')

sys.path.append(project_path)
from util.transformations import *


## **Dim User**

## **AUTOLOADER**


In [0]:
df_user = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option("schemaEvolutionMode", "addNewColumns")\
  .option("cloudFiles.schemaLocation", "abfss://silver@storageabhdatalake.dfs.core.windows.net/DimUser/checkpoints")\
  .load("abfss://bronze@storageabhdatalake.dfs.core.windows.net/DimUser")

In [0]:
# from pyspark.sql.functions import upper, col
df_user = df_user.withColumn("user_name", upper(col("user_name")))
# display(df_user)

In [0]:

df_user_obj = resuable()

df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])
df_user = df_user.dropDuplicates( ['user_id'])
display(df_user)

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageabhdatalake.dfs.core.windows.net/DimUser/checkpoints")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageabhdatalake.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_catalog.silver.DimUser")

## **DimArtist**

In [0]:
df_art = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option("cloudFiles.schemaLocation", "abfss://silver@storageabhdatalake.dfs.core.windows.net/DimArtist/checkpoints")\
  .option("schemaEvolutionMode", "addNewColumns")\
  .load("abfss://bronze@storageabhdatalake.dfs.core.windows.net/DimArtist")

In [0]:
df_artist_obj = resuable()

df_art = df_artist_obj.dropColumns(df_art, ['_rescued_data'])
df_art = df_art.dropDuplicates( ['artist_id'])
display(df_art)


In [0]:
df_art.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageabhdatalake.dfs.core.windows.net/DimArtist/checkpoints")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageabhdatalake.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_catalog.silver.DimArtist")

## **DimTrack**

In [0]:
df_track = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option("cloudFiles.schemaLocation", "abfss://silver@storageabhdatalake.dfs.core.windows.net/DimTrack/checkpoints")\
  .option("schemaEvolutionMode", "addNewColumns")\
  .load("abfss://bronze@storageabhdatalake.dfs.core.windows.net/DimTrack")

In [0]:
display(df_track)

In [0]:
df_track  =df_track.withColumn("durationFlag",when(col("duration_sec")<150, "low")\
    .when(col("duration_sec")<300,"medium")\
    .otherwise("high"))

df_track = df_track.withColumn("track_name",regexp_replace(col("track_name"),"-", " "))

df_track_obj = resuable()

df_track = resuable().dropColumns(df_track, ['_rescued_data'])

display(df_track)

In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageabhdatalake.dfs.core.windows.net/DimTrack/checkpoints")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageabhdatalake.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_catalog.silver.DimTrack")

## DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option("cloudFiles.schemaLocation", "abfss://silver@storageabhdatalake.dfs.core.windows.net/DimDate/checkpoints")\
  .option("schemaEvolutionMode", "addNewColumns")\
  .load("abfss://bronze@storageabhdatalake.dfs.core.windows.net/DimDate")

df_date = resuable().dropColumns(df_date, ['_rescued_data'])

df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageabhdatalake.dfs.core.windows.net/DimDate/checkpoints")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageabhdatalake.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_catalog.silver.DimDate")


## FactStream

In [0]:
df_fact = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "parquet") \
  .option("cloudFiles.schemaLocation", "abfss://silver@storageabhdatalake.dfs.core.windows.net/FactStream/checkpoints")\
  .option("schemaEvolutionMode", "addNewColumns")\
  .load("abfss://bronze@storageabhdatalake.dfs.core.windows.net/FactStream")
df_fact = resuable().dropColumns(df_fact, ['_rescued_data'])
display(df_fact)




In [0]:
df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageabhdatalake.dfs.core.windows.net/FactStream/checkpoints")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageabhdatalake.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_catalog.silver.FactStream")